# 🚀 Week 6 Project: Interactive Sales Dashboard
### Data Visualization Mastery with Seaborn & Plotly

**Dataset:** 100 transactions | 5 Products | 4 Regions | Jan–Apr 2024  
**Tools:** Pandas • Seaborn • Matplotlib • Plotly  

---
| Day | Focus | Output |
|-----|-------|--------|
| Day 1 | Seaborn Basics | Bar + Line charts |
| Day 2 | Statistical Plots | Box + Violin plots |
| Day 3 | Heatmaps | Correlation matrix |
| Day 4 | Multi-plot Grid | 2×2 subplot layout |
| Day 5 | Interactive | Plotly hover/animation |
| Day 6 | Integration | Full dashboard |
| Day 7 | Polish | Final presentation |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings, os
warnings.filterwarnings("ignore")
os.makedirs("visualizations", exist_ok=True)
print("✅ Libraries imported")

## 📂 Load & Explore Data

In [ ]:
df = pd.read_csv("sales_data.csv")
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["Month_Num"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["Day_Name"] = df["Date"].dt.day_name()
print("Shape:", df.shape)
df.head()

In [ ]:
# Data overview
print("=" * 45)
print("DATASET SUMMARY")
print("=" * 45)
print(f"Rows         : {len(df)}")
print(f"Columns      : {df.columns.tolist()}")
print(f"Products     : {df["Product"].unique().tolist()}")
print(f"Regions      : {df["Region"].unique().tolist()}")
print(f"Date Range   : {df["Date"].min().date()} to {df["Date"].max().date()}")
print(f"Total Revenue: ₹{df["Total_Sales"].sum():,.0f}")
df.describe()

## 🎨 Global Theme & Color Palette

In [ ]:
PALETTE = {
    "Phone":      "#4361EE",
    "Laptop":     "#3A0CA3",
    "Tablet":     "#7209B7",
    "Headphones": "#F72585",
    "Monitor":    "#4CC9F0",
}
REGION_PALETTE = ["#4361EE", "#F72585", "#7209B7", "#4CC9F0"]
BG, CARD, TEXT, ACCENT = "#0F0F1A", "#1A1A2E", "#E8E8F0", "#4361EE"

sns.set_theme(style="darkgrid", palette=list(PALETTE.values()))
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": CARD,
    "axes.labelcolor": TEXT, "xtick.color": TEXT,
    "ytick.color": TEXT, "text.color": TEXT,
    "axes.titlecolor": TEXT, "grid.color": "#2A2A3E",
    "axes.edgecolor": "#2A2A3E",
})
print("✅ Theme set")

## 📊 Day 1 — Seaborn Basics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(BG)
fig.suptitle("Sales Overview — Seaborn Basics", fontsize=18, fontweight="bold", color=TEXT)

product_sales = df.groupby("Product")["Total_Sales"].sum().sort_values(ascending=False)
colors = [PALETTE[p] for p in product_sales.index]
bars = axes[0].bar(product_sales.index, product_sales.values/1e6, color=colors)
axes[0].set_title("Total Sales by Product (₹ Millions)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Product"); axes[0].set_ylabel("Total Sales (₹ Millions)")
for bar, val in zip(bars, product_sales.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f"₹{val/1e6:.1f}M", ha="center", fontsize=9, color=TEXT)

monthly = df.groupby("Month")["Total_Sales"].sum().reset_index()
axes[1].plot(monthly["Month"], monthly["Total_Sales"]/1e6, marker="o",
             color=ACCENT, linewidth=2.5, markersize=7, markerfacecolor="#F72585")
axes[1].fill_between(range(len(monthly)), monthly["Total_Sales"]/1e6, alpha=0.15, color=ACCENT)
axes[1].set_xticks(range(len(monthly)))
axes[1].set_xticklabels(monthly["Month"], rotation=30, ha="right")
axes[1].set_title("Monthly Sales Trend (₹ Millions)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Total Sales (₹ Millions)")

plt.tight_layout()
plt.savefig("visualizations/day1_seaborn_basics.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()

## 📊 Day 2 — Statistical Visualizations (Box + Violin)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor(BG)
fig.suptitle("Price & Sales Distribution", fontsize=18, fontweight="bold", color=TEXT)

sns.boxplot(data=df, x="Product", y="Price", palette=PALETTE, width=0.5,
            flierprops=dict(marker="o", markerfacecolor="#F72585", markersize=5), ax=axes[0])
axes[0].set_title("Price Distribution by Product", fontsize=13, fontweight="bold")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"₹{x:,.0f}"))

sns.violinplot(data=df, x="Region", y="Total_Sales", palette=REGION_PALETTE,
               inner="quartile", ax=axes[1])
axes[1].set_title("Total Sales Distribution by Region", fontsize=13, fontweight="bold")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"₹{x/1e3:.0f}K"))

plt.tight_layout()
plt.savefig("visualizations/day2_statistical.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()

## 📊 Day 3 — Heatmaps & Correlation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor(BG)
fig.suptitle("Heatmaps & Correlation", fontsize=18, fontweight="bold", color=TEXT)

corr = df[["Quantity","Price","Total_Sales"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=2, linecolor=BG, square=True,
            annot_kws={"size":14,"weight":"bold"}, ax=axes[0])
axes[0].set_title("Numerical Feature Correlation", fontsize=13, fontweight="bold")

pivot = df.pivot_table(values="Total_Sales", index="Region", columns="Product", aggfunc="mean")
sns.heatmap(pivot/1e3, annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=1.5, linecolor=BG,
            annot_kws={"size":11},
            cbar_kws={"label":"Avg Sales (₹K)"}, ax=axes[1])
axes[1].set_title("Avg Sales (₹K) — Region × Product", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("visualizations/day3_heatmaps.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()

## 📊 Day 4 — Multi-Plot 2×2 Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.patch.set_facecolor(BG)
fig.suptitle("Static Multi-Plot Sales Dashboard", fontsize=20, fontweight="bold", color=TEXT)

# Stacked bar
monthly_product = df.groupby(["Month","Product"])["Total_Sales"].sum().unstack(fill_value=0)
monthly_product.plot(kind="bar", stacked=True, ax=axes[0,0],
                     color=[PALETTE[p] for p in monthly_product.columns])
axes[0,0].set_title("Monthly Sales by Product", fontsize=13, fontweight="bold")
axes[0,0].tick_params(axis="x", rotation=30)
axes[0,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"₹{x/1e6:.1f}M"))

# Count by Region
region_counts = df["Region"].value_counts()
axes[0,1].bar(region_counts.index, region_counts.values, color=REGION_PALETTE)
axes[0,1].set_title("Transaction Count by Region", fontsize=13, fontweight="bold")

# Scatter
for product, color in PALETTE.items():
    sub = df[df["Product"]==product]
    axes[1,0].scatter(sub["Price"], sub["Total_Sales"]/1e3, color=color, label=product, alpha=0.75, s=60)
axes[1,0].set_title("Price vs Total Sales by Product", fontsize=13, fontweight="bold")
axes[1,0].legend(fontsize=8)

# Avg quantity
avg_qty = df.groupby("Product")["Quantity"].mean().sort_values()
axes[1,1].barh(avg_qty.index, avg_qty.values, color=[PALETTE[p] for p in avg_qty.index])
axes[1,1].set_title("Avg Quantity Ordered by Product", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("visualizations/day4_multiplot.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()

## 📊 Day 5 — Interactive Plotly Visualizations

In [ ]:
PLOTLY_THEME = dict(
    template="plotly_dark", paper_bgcolor="#0F0F1A",
    plot_bgcolor="#1A1A2E", font=dict(family="Arial", color="#E8E8F0"),
)

# Interactive line chart
monthly2 = df.groupby(["Month","Product"])["Total_Sales"].sum().reset_index()
fig_line = px.line(monthly2, x="Month", y="Total_Sales", color="Product",
                   markers=True, title="Monthly Sales Trend by Product",
                   color_discrete_map=PALETTE)
fig_line.update_layout(**PLOTLY_THEME)
fig_line.write_html("visualizations/day5_interactive_line.html")
fig_line.show()

In [ ]:
# Interactive scatter with animation
fig_scatter = px.scatter(df, x="Price", y="Total_Sales", color="Product",
                         size="Quantity", animation_frame="Month",
                         title="Price vs Sales Animation",
                         color_discrete_map=PALETTE, size_max=30,
                         hover_name="Customer_ID",
                         hover_data={"Region":True, "Quantity":True})
fig_scatter.update_layout(**PLOTLY_THEME)
fig_scatter.write_html("visualizations/day5_interactive_scatter.html")
fig_scatter.show()

## 📊 Day 6 — Full Interactive Dashboard

In [ ]:
# Run dashboard.py to generate the full Plotly dashboard
import subprocess
result = subprocess.run(["python3", "dashboard.py"], capture_output=True, text=True)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
print("
✅ Open visualizations/day6_full_dashboard.html in your browser!")

## 📊 Day 7 — Key Insights & Business Summary

In [ ]:
print("=" * 55)
print("  📋  KEY BUSINESS INSIGHTS")
print("=" * 55)
print(f"  Total Revenue       : ₹{df["Total_Sales"].sum()/1e6:.2f} Million")
print(f"  Total Transactions  : {len(df)}")
print(f"  Avg Order Value     : ₹{df["Total_Sales"].mean():,.0f}")
print(f"  Top Product         : {df.groupby("Product")["Total_Sales"].sum().idxmax()}")
print(f"  Top Region          : {df.groupby("Region")["Total_Sales"].sum().idxmax()}")
print(f"  Best Month          : {df.groupby("Month")["Total_Sales"].sum().idxmax()}")
print(f"  Avg Price           : ₹{df["Price"].mean():,.0f}")
print(f"  Max Single Order    : ₹{df["Total_Sales"].max():,.0f}")
print("=" * 55)

# Monthly growth
monthly_rev = df.groupby("Month")["Total_Sales"].sum()
growth = ((monthly_rev.iloc[-1] - monthly_rev.iloc[0]) / monthly_rev.iloc[0]) * 100
print(f"  Revenue Growth      : {growth:+.1f}% (Jan→Apr)")
print("=" * 55)